# NYC Parking Violation Risk Analysis

**Tejasree Seelam | ISYE 6740 | Fall 2025**

Exploratory analysis, heuristic risk scoring, and logistic regression using May–October 2025 ticket records. Read the repository README for reported results and methodological limitations.

This publication copy repairs the collection cell syntax and local GeoJSON loading, removes inline installs and stale outputs, and completes the final R-score classifier. The original scoring and exploratory modeling choices remain. The full study CSV was not supplied; this notebook has not been rerun end to end.

Run the setup cell first. If the full CSV is already in the repository root, skip the next two optional collection cells and continue at the pandas load cell. Otherwise run the collection cells explicitly; they make network requests and retain the original daily cap.


In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR
if not DATA_DIR.is_dir():
    raise FileNotFoundError("Start Jupyter from the repository root (see README).")


In [ ]:
# Optional: collects up to 10,000 records per day, matching the original study.
# This is a capped sample, not a complete census. Network failures stop the run.
# Execute only when you want to fetch data; see README for limitations.
import os, time, requests, pandas as pd
from datetime import datetime, timezone
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter

# ---------- CONFIG ----------
DATASET_URL = "https://data.cityofnewyork.us/resource/nc67-uf89.json"
START = pd.Timestamp("2025-05-01")
END   = pd.Timestamp("2025-11-01")       # exclusive upper bound
DAY_LIMIT = 10_000
SELECT_COLS = [
    "summons_number","plate","state","license_type","issue_date","violation_time","violation",
    "fine_amount","penalty_amount","precinct","county","issuing_agency","violation_status"
]
APP_TOKEN = os.getenv("SOCRATA_APP_TOKEN")
TIMEOUT = 60
RETRY = Retry(total=5, backoff_factor=1.5, status_forcelist=[429,500,502,503,504])

# ---------- SESSION ----------
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=RETRY))
headers = {"X-App-Token": APP_TOKEN} if APP_TOKEN else {}

# ---------- HELPERS ----------
def next_month(d):
    return (pd.Timestamp(d.year, d.month, 1) + pd.DateOffset(months=1))

def days_in_range(a, b):
    d = pd.Timestamp(a.date())
    while d < b:
        yield d, d + pd.Timedelta(days=1)
        d += pd.Timedelta(days=1)

# ---------- MAIN ----------
cur = pd.Timestamp(START.year, START.month, 1)
all_month_parts = []

while cur < END:
    m_start, m_end = max(cur, START), min(next_month(cur), END)
    print(f"\n=== Month: {m_start.date()} → {m_end.date()} ===")
    month_rows = []

    for d0, d1 in days_in_range(m_start, m_end):
        day_str = d0.strftime("%m/%d/%Y")
        params = {
            "$select": ",".join(SELECT_COLS),
            "$where": f"issue_date = '{day_str}'",
            "$order": "issue_date, summons_number",
            "$limit": DAY_LIMIT,
        }

        try:
            r = session.get(DATASET_URL, params=params, headers=headers, timeout=TIMEOUT)
            r.raise_for_status()
            batch = r.json()
        except Exception as e:
            raise RuntimeError(f"Download failed on {day_str}; no complete sample was produced") from e

        if not batch:
            continue

        df_day = pd.DataFrame(batch)
        if "issue_date" in df_day.columns:
            df_day["issue_date"] = pd.to_datetime(df_day["issue_date"], format="%m/%d/%Y", errors="coerce")
        month_rows.append(df_day)
        print(f"  {day_str}: {len(df_day):,}/{DAY_LIMIT:,}")
        time.sleep(0.2)

    if month_rows:
        df_month = pd.concat(month_rows, ignore_index=True)
        df_month = df_month[(df_month["issue_date"] >= m_start) & (df_month["issue_date"] < m_end)]
        if "summons_number" in df_month.columns:
            df_month = df_month.drop_duplicates(subset=["summons_number"])
        out_name = f"nyc_parking_{m_start.strftime('%Y-%m')}.csv"
        df_month.to_csv(DATA_DIR / out_name, index=False)
        print(f"Saved {out_name} | {len(df_month):,} rows | {df_month['issue_date'].min()} → {df_month['issue_date'].max()}")
        all_month_parts.append(df_month)

    cur = next_month(cur)

if all_month_parts:
    df_all = pd.concat(all_month_parts, ignore_index=True)
    if "summons_number" in df_all.columns:
        df_all = df_all.drop_duplicates(subset=["summons_number"])
    combined_name = f"nyc_parking_{START.strftime('%Y-%m-%d')}_to_{(END - pd.Timedelta(days=1)).strftime('%Y-%m-%d')}.csv"


In [ ]:
if all_month_parts:
    df_all = pd.concat(all_month_parts, ignore_index=True)
    if "summons_number" in df_all.columns:
        df_all = df_all.drop_duplicates(subset=["summons_number"])
    combined_name = f"nyc_parking_{START.strftime('%Y-%m-%d')}_to_{(END - pd.Timedelta(days=1)).strftime('%Y-%m-%d')}.csv"
    df_all.to_csv(DATA_DIR / combined_name, index=False)
    print(f"\nCombined CSV saved: {combined_name} | {len(df_all):,} rows")
else:
    print("\nNo data retrieved.")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

csv_path = DATA_DIR / "nyc_parking_2025-05-01_to_2025-10-31.csv"
if not csv_path.exists():
    raise FileNotFoundError(
        f"Full study CSV missing: {csv_path}. Run the two optional collection cells "
        "or supply the original full CSV. violations.csv is only one row."
    )
df = pd.read_csv(csv_path)
print(df.shape)
df.head()


In [ ]:
print(df.info())
print(df.isnull().sum().sort_values(ascending=False).head(15))


In [ ]:
for col in ["state","county","vehicle_make","violation"]:
    if col in df.columns:
        print(df[col].value_counts(dropna=False).head())


In [ ]:
df["issue_date"] = pd.to_datetime(df["issue_date"], errors="coerce")
df["month"] = df["issue_date"].dt.month
df["weekday"] = df["issue_date"].dt.day_name()
df["hour"] = pd.to_numeric(df["violation_time"].str[:2], errors="coerce")

# Violations by weekday
sns.countplot(x="weekday", data=df,
              order=["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"])
plt.title("Violations by Day of Week")
plt.xticks(rotation=30)
plt.show()

# Violations by month
sns.countplot(x="month", data=df)
plt.title("Violations by Month (May–Oct 2025)")
plt.show()

# Hour-of-day pattern
sns.histplot(df["hour"], bins=24)
plt.title("Violations by Hour of Day")
plt.xlabel("Hour (approx.)")
plt.show()


In [ ]:
import numpy as np

def parse_violation_hour(v):
    if pd.isna(v):
        return np.nan
    v = str(v).strip().upper()

    # expected formats like '0830A', '1130P', '0930'
    if len(v) < 3:
        return np.nan

    # extract hour part safely
    try:
        hour = int(v[:2])
    except ValueError:
        return np.nan

    # adjust for AM/PM if present
    if 'P' in v and hour < 12:
        hour += 12
    if 'A' in v and hour == 12:
        hour = 0

    # cap to 0–23 range
    if 0 <= hour < 24:
        return hour
    else:
        return np.nan

df["hour"] = df["violation_time"].apply(parse_violation_hour)


In [ ]:
sns.histplot(df["hour"], bins=24)
plt.title("Violations by Hour of Day (Cleaned)")
plt.xlabel("Hour (0=Midnight, 23=11PM)")
plt.show()


In [ ]:
if "state" in df.columns:
    top_states = df["state"].value_counts().head(10)
    sns.barplot(x=top_states.index, y=top_states.values)
    plt.title("Top 10 States for Violations")
    plt.show()

if "license_type" in df.columns:
    sns.countplot(y="license_type", data=df,
                  order=df["license_type"].value_counts().head(10).index)
    plt.title("License Types with Most Violations")
    plt.show()


In [ ]:
if "precinct" in df.columns:
    df["precinct"] = pd.to_numeric(df["precinct"], errors="coerce")
    top_precincts = df["precinct"].value_counts().head(15)
    sns.barplot(x=top_precincts.index, y=top_precincts.values)
    plt.title("Top 15 Precincts by Violation Count")
    plt.xlabel("Precinct")
    plt.ylabel("Count")
    plt.show()

if "county" in df.columns:
    sns.countplot(y="county", data=df,
                  order=df["county"].value_counts().index)
    plt.title("Violations by County")
    plt.show()


In [ ]:
# Convert to numeric, but keep NaN for non-numeric entries
df["precinct"] = pd.to_numeric(df["precinct"], errors="coerce")

# Drop or replace missing precincts (0 usually means missing)
df = df[df["precinct"].notna() & (df["precinct"] != 0)]

# Cast to integer for nicer labels
df["precinct"] = df["precinct"].astype(int)


In [ ]:
top_precincts = df["precinct"].value_counts().head(15)

plt.figure(figsize=(10,5))
sns.barplot(x=top_precincts.index.astype(str), y=top_precincts.values)
plt.title("Top 15 Precincts by Violation Count")
plt.xlabel("Precinct")
plt.ylabel("Number of Violations")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
df["fine_amount"] = pd.to_numeric(df["fine_amount"], errors="coerce")
df["penalty_amount"] = pd.to_numeric(df["penalty_amount"], errors="coerce")

# Replace missing with 0 for analysis
df[["fine_amount", "penalty_amount"]] = df[["fine_amount", "penalty_amount"]].fillna(0)

df["total_due"] = df["fine_amount"] + df["penalty_amount"]

print(df[["fine_amount", "penalty_amount", "total_due"]].describe())


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
sns.histplot(df["total_due"], bins=50)
plt.title("Distribution of Total Fine + Penalty Amounts")
plt.xlabel("Total Amount ($)")
plt.xlim(0, 500)  # cap to keep visualization reasonable
plt.show()

plt.figure(figsize=(8,5))
sns.boxplot(x=df["total_due"])
plt.title("Fine + Penalty Distribution (Boxplot)")
plt.xlabel("Total Due ($)")
plt.show()


In [ ]:
# feature engineering
import numpy as np
import pandas as pd

# Dates & time
df["issue_date"] = pd.to_datetime(df["issue_date"], errors="coerce")
df["weekday"] = df["issue_date"].dt.day_name()
df["is_weekend"] = df["weekday"].isin(["Saturday","Sunday"]).astype(int)

# Hour (clean parse with AM/PM noise handled)
def parse_violation_hour(v):
    if pd.isna(v): return np.nan
    s = str(v).strip().upper()
    if len(s) < 3: return np.nan
    try: h = int(s[:2])
    except: return np.nan
    if 'P' in s and h < 12: h += 12
    if 'A' in s and h == 12: h = 0
    return h if 0 <= h < 24 else np.nan

df["hour"] = df["violation_time"].apply(parse_violation_hour)

# Peak hours (adjust if your EDA suggests a different window)
df["is_peak"] = df["hour"].between(8,18, inclusive="left").astype(int)


In [ ]:
df["precinct"] = pd.to_numeric(df["precinct"], errors="coerce").astype("Int64")
area_key = ["county","precinct"]    # use both to reduce ID ambiguity


In [ ]:
# Violation density score - rolling history

# daily counts per area
daily = (df
         .groupby(area_key + [df["issue_date"].dt.date.rename("day")])
         .size()
         .reset_index(name="cnt"))

# 30-day rolling sum per area (requires full date index)
daily["day"] = pd.to_datetime(daily["day"])
daily = (daily
         .set_index("day")
         .groupby(area_key)["cnt"]
         .apply(lambda s: s.asfreq("D", fill_value=0).rolling(30, min_periods=7).sum())
         .reset_index())

# normalize to 0–1 per area (min-max over the window)
density = (daily
           .groupby(area_key)["cnt"]
           .transform(lambda x: (x - x.min())/(x.max() - x.min() + 1e-9)))
daily["density_score"] = density

# attach back to tickets (left-join on area+day)
df["day"] = df["issue_date"].dt.floor("D")
df = df.merge(daily[area_key+["day","density_score"]], on=area_key+["day"], how="left")
df["density_score"] = df["density_score"].fillna(0.0)


In [ ]:
# camera score - based on camera types
camera_like = df["violation"].str.contains("SPEED|BUS LANE|RED LIGHT", case=True, na=False)
cam_daily = (df.assign(camera_like=camera_like)
               .groupby(area_key+["day"])["camera_like"].mean().reset_index())
# smooth + normalize per area
cam_daily["camera_like"] = cam_daily.groupby(area_key)["camera_like"]\
                                    .transform(lambda s: s.rolling(30, min_periods=7).mean())
cam_daily["camera_score"] = cam_daily.groupby(area_key)["camera_like"]\
                                     .transform(lambda x: (x - x.min())/(x.max()-x.min()+1e-9))
df = df.merge(cam_daily[area_key+["day","camera_score"]], on=area_key+["day"], how="left")
df["camera_score"] = df["camera_score"].fillna(0.0)


In [ ]:
#driver persona features, out of state, license type

df["is_out_of_state"] = (df["state"].str.upper() != "NY").astype(int)

def license_bucket(s):
    s = str(s).upper()
    if s in {"PAS","SRF","SRN"}: return "passenger"
    if s in {"COM","OMT","MED"}: return "commercial"
    return "other"
df["license_bucket"] = df["license_type"].apply(license_bucket)
persona_dummies = pd.get_dummies(df["license_bucket"], prefix="lic", dummy_na=False)
df = pd.concat([df, persona_dummies], axis=1)


In [ ]:
#severity proxy

for c in ["fine_amount","penalty_amount"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
df["total_due"] = df["fine_amount"] + df["penalty_amount"]

# define "high severity" if total_due >= threshold (tune via EDA)
SEV_THR = 150
df["is_high_severity"] = (df["total_due"] >= SEV_THR).astype(int)


In [ ]:
#heuristic R-score

# time score: peak + weekday
df["time_score"] = 0.7*df["is_peak"] + 0.3*(1 - df["is_weekend"])
# normalize within [0,1]
# already 0–1, but keep the pattern if you expand
df["time_score"] = df["time_score"]

# persona score: out-of-state + license type
df["persona_score"] = 0.7*df["is_out_of_state"] + 0.3*df.get("lic_commercial", 0)

# R = 0.45 camera + 0.35 density + 0.15 time + persona adj (capped)
R = (0.45*df["camera_score"] +
     0.35*df["density_score"] +
     0.15*df["time_score"] +
     0.10*df["persona_score"])

df["R_raw"] = R
df["R"] = df["R_raw"].clip(0,1)

# Risk tiers
def tier(r):
    if r >= 0.70: return "High"
    if r >= 0.40: return "Medium"
    return "Low"
df["risk_tier"] = df["R"].apply(tier)


In [ ]:
#top 3 local violation types

top3 = (df.groupby(area_key + ["violation"])
          .size()
          .reset_index(name="n")
          .sort_values(["county","precinct","n"], ascending=[True,True,False])
          .groupby(area_key)
          .head(3))

# attach as a comma list per area
top3_list = (top3.groupby(area_key)["violation"]
                  .apply(lambda s: ", ".join(s.head(3)))
                  .reset_index(name="top3_violations"))
df = df.merge(top3_list, on=area_key, how="left")


In [ ]:
agg = (df.groupby(area_key + ["day"])
         .agg(
             tickets=("summons_number","count"),
             mean_R=("R","mean"),
             pct_peak=("is_peak","mean"),
             pct_out_of_state=("is_out_of_state","mean"),
             mean_fine=("fine_amount","mean"),
             mean_penalty=("penalty_amount","mean"),
             high_sev_rate=("is_high_severity","mean")
         ).reset_index())

# Binary target: high severity day if rate ≥ 0.2 (tune)
agg["y_high_sev_day"] = (agg["high_sev_rate"] >= 0.20).astype(int)


In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, brier_score_loss

features = ["mean_R","pct_peak","pct_out_of_state","tickets","mean_fine","mean_penalty"]
X = agg[features].fillna(0).values
y = agg["y_high_sev_day"].values

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=200, class_weight="balanced"))
])

# simple time-based split: first 70% days train, rest test
agg_sorted = agg.sort_values("day")
cut = int(len(agg_sorted)*0.7)
Xtr, Xte = agg_sorted[features].fillna(0).values[:cut], agg_sorted[features].fillna(0).values[cut:]
ytr, yte = agg_sorted["y_high_sev_day"].values[:cut], agg_sorted["y_high_sev_day"].values[cut:]

pipe.fit(Xtr, ytr)
pred = pipe.predict_proba(Xte)[:,1]

print("AUC:", roc_auc_score(yte, pred))
print("Brier:", brier_score_loss(yte, pred))


In [ ]:
from sklearn.calibration import CalibratedClassifierCV
#cal = CalibratedClassifierCV(base_estimator=pipe.named_steps["lr"], method="isotonic", cv=3)
cal = CalibratedClassifierCV(pipe.named_steps["lr"], method="isotonic", cv=3)

# fit on scaled features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(Xtr)
cal.fit(scaler.transform(Xtr), ytr)
pred_cal = cal.predict_proba(scaler.transform(Xte))[:,1]
print("AUC (calibrated):", roc_auc_score(yte, pred_cal))
print("Brier (calibrated):", brier_score_loss(yte, pred_cal))


In [ ]:
latest = (df.sort_values("day").groupby(area_key).tail(1)
          [area_key+["day","R","risk_tier","top3_violations"]]
          .rename(columns={"R":"R_today"}))
latest.head()


In [ ]:
#make sure each county and precint has one latest row and sorted by risk
out = latest.sort_values("R_today", ascending=False)  # ‘latest’ from earlier step
out.head(30)


In [ ]:
# recompute top-3 properly
vc = (df.groupby(["county","precinct","violation"])
        .size()
        .reset_index(name="n"))
vc = vc.sort_values(["county","precinct","n"], ascending=[True, True, False])

top3 = (vc.groupby(["county","precinct"])
          .head(3)
          .groupby(["county","precinct"])["violation"]
          .apply(lambda s: ", ".join(s.tolist()))
          .reset_index(name="top3_violations"))

# attach to your latest rows
latest = latest.drop(columns=["top3_violations"], errors="ignore")\
               .merge(top3, on=["county","precinct"], how="left")
latest.head()

In [ ]:
#risk board - hot areas to include in the report
risk_board = (latest
              .assign(precinct=latest["precinct"].astype("Int64"))
              .loc[:, ["county","precinct","R_today","risk_tier","top3_violations"]]
              .sort_values("R_today", ascending=False))
risk_board.head(25)


In [ ]:
# aggregate by area×day to compare R and severity
eval_tbl = (df.groupby(["county","precinct","day"])
              .agg(mean_R=("R","mean"),
                   high_sev_rate=("is_high_severity","mean"),
                   tickets=("summons_number","count"))
              .reset_index())

eval_tbl["R_bin"] = pd.qcut(eval_tbl["mean_R"], 5, labels=False, duplicates="drop")
eval_tbl.groupby("R_bin")["high_sev_rate"].mean()  # should be monotonically increasing-ish


In [ ]:
import pandas as pd
from datetime import date

def risk_for(county:str, precinct:int, on_day:pd.Timestamp|None=None):
    sub = df[(df["county"]==county) & (df["precinct"]==precinct)]
    if sub.empty:
        return {"error": "No data for that county/precinct."}
    if on_day is None:
        on_day = sub["day"].max()
    row = (sub[sub["day"]==on_day]
           .sort_values("issue_date")
           .tail(1)[["R","risk_tier"]])
    if row.empty:
        return {"error": "No records on that day for the area."}
    r = row.iloc[0].to_dict()
    # attach top-3 (precomputed ‘top3’ above)
    t3 = top3[(top3["county"]==county) & (top3["precinct"]==precinct)]
    r["top3_violations"] = t3["top3_violations"].iloc[0] if not t3.empty else None
    r["county"], r["precinct"], r["day"] = county, int(precinct), pd.to_datetime(on_day).date()
    return r

# example
risk_for("Q", 23)


In [ ]:
df[["R","is_high_severity","density_score","camera_score","time_score"]].corr()


In [ ]:
df["density_score"] = 1 - df["density_score"]
df["time_score"] = 1 - df["time_score"]  # optional, check after flipping density


In [ ]:
df["R_fixed"] = (
    0.45*df["camera_score"] +
    0.35*df["density_score"] +
    0.15*df["time_score"] +
    0.10*df["persona_score"]
).clip(0,1)

df[["R_fixed","is_high_severity","density_score","camera_score","time_score"]].corr()


In [ ]:
#r binned trend plot - shows an upward trend

df["R_bin"] = pd.qcut(df["R_fixed"], 5, labels=False)
mean_sev = df.groupby("R_bin")["is_high_severity"].mean()
mean_sev.plot(marker="o")
plt.title("High Severity Rate vs R quintile (after correction)")
plt.xlabel("R quintile (0=Low Risk → 4=High Risk)")
plt.ylabel("Mean High-Severity Rate")
plt.show()


In [ ]:
def tier(r):
    if r >= 0.70: return "High"
    if r >= 0.40: return "Medium"
    return "Low"

df["risk_tier"] = df["R_fixed"].apply(tier)


In [ ]:
risk_summary = (df.groupby(["county","precinct"])["R_fixed"]
                  .mean()
                  .reset_index()
                  .sort_values("R_fixed", ascending=False))
risk_summary.head(10)


In [ ]:
# Install dependencies once with: python -m pip install -r requirements.txt


In [ ]:
import json
import geopandas as gpd

geojson_path = DATA_DIR / "nyc_police_precincts.geojson"
with geojson_path.open() as f:
    gj = json.load(f)
gdf = gpd.read_file(geojson_path)
gdf.head()


In [ ]:
type(gj), list(gj.keys())



In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
X = df[["R_fixed"]]  # predictor
y = df["is_high_severity"]  # target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Complete the R-score-only baseline described in the report.
r_model = LogisticRegression(max_iter=1000)
r_model.fit(X_train, y_train)
y_pred = r_model.predict(X_test)
y_prob = r_model.predict_proba(X_test)[:, 1]
print("Coefficient:", r_model.coef_[0, 0])
print("Intercept:", r_model.intercept_[0])
print(classification_report(y_test, y_pred, zero_division=0))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.plot(fpr, tpr, label="R-score logistic regression")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("High-severity classification ROC")
plt.legend()
plt.show()
